In [11]:
#importing all necessary libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [12]:
#loading the dataset and splitting into training and testing sets
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [13]:
#scaling data
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

In [14]:
#list of models that need to be compared
models = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'SVM': SVC()
}

In [15]:
#function to generate different evaluation metrics of different models
def evaluate_models(X_train, X_test, title):
    print(f'\n===== {title} =====')
    results = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='macro')
        rec = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')
        cm = confusion_matrix(y_test, y_pred)
        print(f'\n{name}')
        print(f'Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1-Score: {f1:.4f}')
        print('Confusion Matrix:\n', cm)
        results.append({
            'Model': name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-Score': f1
        })
    return pd.DataFrame(results)

In [16]:
#evaluating no restriction models
results_no_reduction = evaluate_models(X_train_std, X_test_std, 'Original Data (No Dimensionality Reduction)')


===== Original Data (No Dimensionality Reduction) =====

Logistic Regression
Accuracy: 0.9111 | Precision: 0.9155 | Recall: 0.9111 | F1-Score: 0.9107
Confusion Matrix:
 [[15  0  0]
 [ 0 14  1]
 [ 0  3 12]]

Decision Tree
Accuracy: 0.9111 | Precision: 0.9111 | Recall: 0.9111 | F1-Score: 0.9111
Confusion Matrix:
 [[15  0  0]
 [ 0 13  2]
 [ 0  2 13]]

Random Forest
Accuracy: 0.8889 | Precision: 0.8981 | Recall: 0.8889 | F1-Score: 0.8878
Confusion Matrix:
 [[15  0  0]
 [ 0 14  1]
 [ 0  4 11]]

SVM
Accuracy: 0.9333 | Precision: 0.9345 | Recall: 0.9333 | F1-Score: 0.9333
Confusion Matrix:
 [[15  0  0]
 [ 0 14  1]
 [ 0  2 13]]


In [17]:
#evaluating pca metrics
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca = pca.transform(X_test_std)
results_pca = evaluate_models(X_train_pca, X_test_pca, 'PCA-Reduced Data')


===== PCA-Reduced Data =====

Logistic Regression
Accuracy: 0.8889 | Precision: 0.8981 | Recall: 0.8889 | F1-Score: 0.8878
Confusion Matrix:
 [[15  0  0]
 [ 0 14  1]
 [ 0  4 11]]

Decision Tree
Accuracy: 0.9111 | Precision: 0.9155 | Recall: 0.9111 | F1-Score: 0.9107
Confusion Matrix:
 [[15  0  0]
 [ 0 14  1]
 [ 0  3 12]]

Random Forest
Accuracy: 0.9333 | Precision: 0.9345 | Recall: 0.9333 | F1-Score: 0.9333
Confusion Matrix:
 [[15  0  0]
 [ 0 14  1]
 [ 0  2 13]]

SVM
Accuracy: 0.9111 | Precision: 0.9155 | Recall: 0.9111 | F1-Score: 0.9107
Confusion Matrix:
 [[15  0  0]
 [ 0 14  1]
 [ 0  3 12]]


In [18]:
#evaluating lda metrics
lda = LDA(n_components=2)
X_train_lda = lda.fit_transform(X_train_std, y_train)
X_test_lda = lda.transform(X_test_std)
results_lda = evaluate_models(X_train_lda, X_test_lda, 'LDA-Reduced Data')


===== LDA-Reduced Data =====

Logistic Regression
Accuracy: 0.9778 | Precision: 0.9792 | Recall: 0.9778 | F1-Score: 0.9778
Confusion Matrix:
 [[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]

Decision Tree
Accuracy: 0.9778 | Precision: 0.9792 | Recall: 0.9778 | F1-Score: 0.9778
Confusion Matrix:
 [[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]

Random Forest
Accuracy: 0.9778 | Precision: 0.9792 | Recall: 0.9778 | F1-Score: 0.9778
Confusion Matrix:
 [[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]

SVM
Accuracy: 0.9778 | Precision: 0.9792 | Recall: 0.9778 | F1-Score: 0.9778
Confusion Matrix:
 [[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]


In [19]:
#printing out final values
print('\nComparison of Model Performance Across Techniques =====')
combined = results_no_reduction.copy()
combined['Scenario'] = 'Original'
combined = pd.concat([
    combined,
    results_pca.assign(Scenario='PCA'),
    results_lda.assign(Scenario='LDA')
])
print(combined.pivot_table(index='Model', columns='Scenario', values='Accuracy'))


Comparison of Model Performance Across Techniques =====
Scenario                  LDA  Original       PCA
Model                                            
Decision Tree        0.977778  0.911111  0.911111
Logistic Regression  0.977778  0.911111  0.888889
Random Forest        0.977778  0.888889  0.933333
SVM                  0.977778  0.933333  0.911111
